# 01 — EDA dan Pemeriksaan Dataset

Notebook ini dipakai untuk memahami isi dataset sebelum preprocessing.
- bentuk dan kolom open dataset
- bentuk dan kolom self-construction dataset
- distribusi label
- status kode pada open dataset
- bahasa pemrograman yang tersedia

In [1]:
!pip -q install datasets

from datasets import load_dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import re
import os
import sys

from datasets import load_dataset

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/FP AI"
sys.path.append(os.path.join(PROJECT_DIR, "src"))

from config import *

print("Project directory:")
print(PROJECT_DIR)

Mounted at /content/drive
Project directory:
/content/drive/MyDrive/FP AI


In [4]:
dataset = load_dataset("basakdemirok/AIGCodeSet")

print(dataset)
print("Available splits:", list(dataset.keys()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

data/all_data_with_ada_embeddings_will_b(…):   0%|          | 0.00/265M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7583 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7583 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['problem_id', 'submission_id', 'status_in_folder', 'LLM', 'code', 'ada_embedding', 'label', 'lines', 'code_lines', 'comments', 'functions', 'blank_lines'],
        num_rows: 7583
    })
    test: Dataset({
        features: ['problem_id', 'submission_id', 'status_in_folder', 'LLM', 'code', 'ada_embedding', 'label', 'lines', 'code_lines', 'comments', 'functions', 'blank_lines'],
        num_rows: 7583
    })
})
Available splits: ['train', 'test']


In [5]:
open_parts = []

for split_name in dataset.keys():
    temp_df = dataset[split_name].to_pandas()
    temp_df["hf_split"] = split_name
    open_parts.append(temp_df)

open_df = pd.concat(open_parts, ignore_index=True)

print("Open dataset shape:", open_df.shape)
display(open_df.head())

Open dataset shape: (15166, 13)


,problem_id,submission_id,status_in_folder,LLM,code,ada_embedding,label,lines,code_lines,comments,functions,blank_lines,hf_split
0,p03243,s556163757,Runtime,GEMINI,N = int(input())\nif N % 111 == 0:\n print(N)\nelse:\n for i in range(10):\n if N // 111 == i:\n print(111...,"[0.009706709533929825, 2.121167017321568e-05, 0.009275741875171661, 0.004617980215698481, 0.009713339619338512, 0.00...",1,7,7,0,0,0,train
1,p02949,unrelated,Generate,GEMINI,"from collections import defaultdict\n\ndef solve():\n n, m, p = map(int, input().split())\n graph = defaultdic...","[0.04156259819865227, -0.01601836271584034, -0.004842760972678661, -0.020954255014657974, -0.019158175215125084, 0.0...",1,40,32,4,1,8,train
2,p02701,s743897659,Wrong,LLAMA,N = int(input())\na = []\n\nfor i in range(N):\n a.append(input())\n \nb = set(a)\nprint(len(b)),"[0.011799769476056099, -0.0023118883837014437, 0.023775553330779076, 0.007616030052304268, -0.014690476469695568, 0....",1,8,6,0,0,2,train
3,p03380,s136562567,Runtime,GEMINI,"import math\nimport numpy\n\nn=int(input())\na=list(map(int,input().split()))\n\nmax_num=max(a)\nmiddle_num=math.cei...","[0.027964850887656212, -0.001971880439668894, 0.0025234553031623363, -0.018491549417376518, -0.018174394965171814, 0...",1,26,22,0,0,4,train
4,p03229,unrelated,Generate,LLAMA,"def max_sum_of_abs_diffs(arr):\n arr.sort()\n return sum(abs(arr[i] - arr[i - 1]) for i in range(1, len(arr)))","[0.009082085452973843, 0.011663861572742462, 0.03381209075450897, -0.036669082939624786, -0.007391480728983879, 0.02...",1,3,3,0,1,0,train


In [6]:
self_df = pd.read_excel(SELF_DATASET_FILE)

print("Self-construction dataset shape:", self_df.shape)
display(self_df.head())

Self-construction dataset shape: (324, 8)


,problem_id,problem_title,prompt,code,label,source,language,Jenis LLM
0,sc001,3Sum,Write a Python function to solve the following LeetCode problem.\n Return only the source code without explanation.\...,"class Solution:\n # @return a list of lists of length 3, [[val1,val2,val3]]\n def threeSum(self, num):\n num.sort...",0,swm8023/leetcode-solution,Python,ChatGPT
1,sc001_ai,3Sum,Write a Python function to solve the following LeetCode problem.\n Return only the source code without explanation.\...,"class Solution:\ndef threeSum(self,nums):\nnums.sort();r=[];n=len(nums)\nfor i in range(n-2):\nif i and nums[i]==num...",1,AI-generated (controlled prompting),Python,ChatGPT
2,sc002,3Sum Closest,Write a Python function to solve the following LeetCode problem.\n Return only the source code without explanation.\...,"class Solution:\n # @return an integer\n def threeSumClosest(self, num, target):\n num.sort()\n ans = None\n fo...",0,swm8023/leetcode-solution,Python,ChatGPT
3,sc002_ai,3Sum Closest,Write a Python function to solve the following LeetCode problem.\n Return only the source code without explanation.\...,"class Solution:\ndef threeSumClosest(self,nums,target):\nnums.sort();n=len(nums);a=nums[0]+nums[1]+nums[2]\nfor i in...",1,AI-generated (controlled prompting),Python,ChatGPT
4,sc003,4Sum,Write a Python function to solve the following LeetCode problem.\n Return only the source code without explanation.\...,"class Solution:\n # @return a list of lists of length 4, [[val1,val2,val3,val4]]\n def fourSum(self, num, target):...",0,zhaochuanshen/leetcode,Python,ChatGPT


In [7]:
print("Open dataset columns:")
print(open_df.columns.tolist())

print("\nSelf-construction dataset columns:")
print(self_df.columns.tolist())

Open dataset columns:
['problem_id', 'submission_id', 'status_in_folder', 'LLM', 'code', 'ada_embedding', 'label', 'lines', 'code_lines', 'comments', 'functions', 'blank_lines', 'hf_split']

Self-construction dataset columns:
['problem_id', 'problem_title', 'prompt', 'code', 'label', 'source', 'language', 'Jenis LLM']


In [8]:
def find_possible_columns(df, keywords):
    result = []
    for col in df.columns:
        col_lower = col.lower()
        for keyword in keywords:
            if keyword.lower() in col_lower:
                result.append(col)
                break
    return result

open_code_cols = find_possible_columns(open_df, ["code", "source"])
open_label_cols = find_possible_columns(open_df, ["label", "class", "target"])
open_status_cols = find_possible_columns(open_df, ["status", "verdict", "result", "wrong", "accepted", "correct"])
open_language_cols = find_possible_columns(open_df, ["language", "lang"])
open_problem_cols = find_possible_columns(open_df, ["problem", "task"])

print("Possible code columns:", open_code_cols)
print("Possible label columns:", open_label_cols)
print("Possible status columns:", open_status_cols)
print("Possible language columns:", open_language_cols)
print("Possible problem columns:", open_problem_cols)

Possible code columns: ['code', 'code_lines']
Possible label columns: ['label']
Possible status columns: ['status_in_folder']
Possible language columns: []
Possible problem columns: ['problem_id']


In [9]:
if "label" in open_df.columns:
    print("Open dataset label distribution:")
    display(open_df["label"].value_counts(dropna=False))

    print("\nOpen dataset label distribution percentage:")
    display(open_df["label"].value_counts(normalize=True, dropna=False))
else:
    print("Column 'label' not found. Check possible label columns above.")

Open dataset label distribution:


,count
label,
0,9510
1,5656



Open dataset label distribution percentage:


,proportion
label,
0,0.627061
1,0.372939


In [10]:
if "label" in self_df.columns:
    print("Self-construction label distribution:")
    display(self_df["label"].value_counts(dropna=False))

    print("\nSelf-construction label distribution percentage:")
    display(self_df["label"].value_counts(normalize=True, dropna=False))
else:
    print("Column 'label' not found in self dataset.")

Self-construction label distribution:


,count
label,
0,162
1,162



Self-construction label distribution percentage:


,proportion
label,
0,0.5
1,0.5


In [11]:
print("Self dataset language distribution:")

if "language" in self_df.columns:
    display(self_df["language"].value_counts(dropna=False))
else:
    print("No language column in self dataset.")

print("\nOpen dataset possible language columns:")
print(open_language_cols)

for col in open_language_cols:
    print(f"\nOpen dataset language distribution from column: {col}")
    display(open_df[col].value_counts(dropna=False).head(30))

Self dataset language distribution:


,count
language,
Python,324



Open dataset possible language columns:
[]


In [12]:
print("Possible status columns:")
print(open_status_cols)

for col in open_status_cols:
    print(f"\nStatus distribution from column: {col}")
    display(open_df[col].value_counts(dropna=False).head(50))

Possible status columns:
['status_in_folder']

Status distribution from column: status_in_folder


,count
status_in_folder,
Wrong,5062
Runtime,5052
Accepted,3170
Generate,1882
